# Design of Experiments (DOE)

This notebook demonstates use of Design Of Experiment (DOE) capabilities included in the ProcessOptimizer code.

First, we define the search space that we are interest in like we would for a normal optimization problem.
We do this using the Space class from the ProcessOptimizer library.

Note that we have also included a 2-level categorical variable.
At current, the version of D-optimal designs in ProcessOptimizer does not support categorical variables with more than two levels.

In [2]:
import numpy as np

from ProcessOptimizer.space import Categorical, Integer, Real, Space

factor_space = Space(dimensions=[
    Real(10, 40, name='var_x1'),
    Integer(20, 100, name='var_x2'),
    Integer(-30, 30, name='var_x3'),
    Categorical(['a', 'b'], name='var_x4')
    ])

## D-optimal design

D-optimal design is a type of experimental design that is used to find the optimal threatment combinations for a given number of experiments.

Besides the search space, we also need to define the number of experiments that we want to run and the type of model we want to be able to fit to the data.

In [3]:
from ProcessOptimizer.samplers.doe import get_optimal_DOE

number_of_experiments = 12

design, factor_names = get_optimal_DOE(factor_space, number_of_experiments)

print("Factor names:")
print(factor_names)
print("Design:")
print(design)

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[[np.float64(10.0) np.int64(100) np.int64(30) 'a']
 [np.float64(10.0) np.int64(20) np.int64(-30) 'a']
 [np.float64(10.0) np.int64(20) np.int64(30) 'a']
 [np.float64(40.0) np.int64(100) np.int64(30) 'b']
 [np.float64(10.0) np.int64(100) np.int64(-30) 'b']
 [np.float64(10.0) np.int64(20) np.int64(-30) 'b']
 [np.float64(40.0) np.int64(100) np.int64(-30) 'a']
 [np.float64(40.0) np.int64(20) np.int64(-30) 'b']
 [np.float64(10.0) np.int64(100) np.int64(30) 'b']
 [np.float64(40.0) np.int64(100) np.int64(-30) 'b']
 [np.float64(40.0) np.int64(20) np.int64(30) 'b']
 [np.float64(40.0) np.int64(20) np.int64(30) 'a']]


This is consistent with the type of each dimension in the search space, but not particularly print friendly. Thus we convert the numbers to strings and get:

In [4]:
print_friendly_design = np.asarray(design, dtype=str)

print("Factor names:")
print(factor_names)
print("Design:")
print(print_friendly_design)

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[['10.0' '100' '30' 'a']
 ['10.0' '20' '-30' 'a']
 ['10.0' '20' '30' 'a']
 ['40.0' '100' '30' 'b']
 ['10.0' '100' '-30' 'b']
 ['10.0' '20' '-30' 'b']
 ['40.0' '100' '-30' 'a']
 ['40.0' '20' '-30' 'b']
 ['10.0' '100' '30' 'b']
 ['40.0' '100' '-30' 'b']
 ['40.0' '20' '30' 'b']
 ['40.0' '20' '30' 'a']]


Besides the design space and number of experiments, a number of other parameters can be set. These are:
- `design_type` : Various keywords can be used to specify the regression model that the design should be optimal for.
- `model` : As an alternative to `design_type`, a model can be specified directly. `design_type` and `model` are mutually exclusive. If both are specified, `design_type` will be used.
- `replicates` : A number of replicates can be specified. This is useful when the same experiment is to be run multiple times.
- `sorting` :  Various keywords can be used to specify sorting of experiments in the design.
- `res` : A resolution can be specified. This controls how coarsely the factors should be sampled during the design optimization. 

## design_type

`design_type` can be set to one of the following:
- `linear` : Simple linear regression model without interaction terms.
- `screening` : Screening design with main effects and two-factor interactions.This will be the default if no `design_type` or `model` is specified.
- `response` : Response surface design with main effects, two-factor interactions, and quadratic effects for non-categorical factors.
- `optimization` : Optimization design with main effects, two- and three-factor interactions, and quadratic and cubic effects for non-categorical factors.

Lets see the difference when we use a different one than the screening design, which we used before, as it is the default.

If we keep the experimental budget the same, we see that we get an error. We cannot fit a model with 14 parameters using only 12 experiments.

In [5]:
number_of_experiments = 12

design, factor_names = get_optimal_DOE(factor_space, number_of_experiments, design_type='response')
print_friendly_design = np.asarray(design, dtype=str)

print("Factor names:")
print(factor_names)
print("Design:")
print(print_friendly_design)

ValueError: Can't build a design of size 12 for a model of rank 14. Model: '(var_x1+var_x2+var_x3+var_x4)**2+pow(var_x1, 2)+pow(var_x2, 2)+pow(var_x3, 2)'

Let us try increase the budget a bit.

In [6]:
number_of_experiments = 15

design, factor_names = get_optimal_DOE(factor_space, number_of_experiments, design_type='response')
print_friendly_design = np.asarray(design, dtype=str)

print("Factor names:")
print(factor_names)
print("Design:")
print(print_friendly_design)

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[['25.0' '52' '30' 'a']
 ['40.0' '36' '-30' 'a']
 ['40.0' '100' '-30' 'b']
 ['10.0' '20' '30' 'b']
 ['40.0' '76' '30' 'b']
 ['16.0' '100' '12' 'b']
 ['40.0' '20' '0' 'b']
 ['40.0' '20' '30' 'a']
 ['10.0' '20' '-30' 'a']
 ['10.0' '60' '-30' 'b']
 ['10.0' '100' '30' 'a']
 ['10.0' '44' '6' 'a']
 ['13.0' '100' '-30' 'a']
 ['40.0' '100' '12' 'a']
 ['25.0' '20' '-30' 'b']]


Note, that in the `screening` design we primarily sample the corners of the search space, while in the `response` design we also sample values of the numerical dimensions, that are near the center of the specified range to fit the quadratic effects.

## res

Lets next change the `res` parameter to see how that affects the design. Default resolution is 11. We generally want to use odd rather than even numbers for resolution. This ensures that we sample the center point of the numerical dimensions. 

In [7]:
number_of_experiments = 15
resolution = 3

design, factor_names = get_optimal_DOE(factor_space,
                                       number_of_experiments,
                                       design_type='response',
                                       res=resolution)
print_friendly_design = np.asarray(design, dtype=str)

print("Factor names:")
print(factor_names)
print("Design:")
print(print_friendly_design)

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[['10.0' '20' '-30' 'b']
 ['40.0' '20' '30' 'a']
 ['40.0' '20' '30' 'b']
 ['40.0' '100' '0' 'a']
 ['40.0' '100' '30' 'b']
 ['10.0' '20' '30' 'b']
 ['10.0' '100' '30' 'a']
 ['10.0' '100' '0' 'b']
 ['10.0' '20' '0' 'a']
 ['40.0' '60' '0' 'b']
 ['40.0' '20' '-30' 'a']
 ['25.0' '60' '30' 'a']
 ['10.0' '100' '-30' 'a']
 ['25.0' '20' '0' 'b']
 ['40.0' '100' '-30' 'b']]


With a resolution of 3, we see that that only values for each dimension present in the design are the minimum, maximum and center values.

In [8]:
number_of_experiments = 15
resolution = 31

design, factor_names = get_optimal_DOE(factor_space,
                                       number_of_experiments,
                                       design_type='response',
                                       res=resolution)
print_friendly_design = np.asarray(design, dtype=str)

print("Factor names:")
print(factor_names)
print("Design:")
print(print_friendly_design)

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[['10.0' '20' '-30' 'a']
 ['10.0' '20' '30' 'a']
 ['40.0' '20' '-30' 'a']
 ['40.0' '63' '30' 'a']
 ['28.0' '20' '4' 'a']
 ['10.0' '84' '-4' 'a']
 ['40.0' '100' '8' 'b']
 ['10.0' '100' '-30' 'b']
 ['40.0' '20' '30' 'b']
 ['10.0' '20' '-8' 'b']
 ['24.0' '100' '30' 'a']
 ['23.0' '57' '30' 'b']
 ['10.0' '100' '30' 'b']
 ['40.0' '100' '-30' 'a']
 ['40.0' '39' '-30' 'b']]


Using a resolution of 31, we see that the factor values can be more varied.
The minimum, maximum and center values are still the most common. 

Rerunning any of the above cells will generate a new design. The result is not deterministic.

The design will converge from the random initial design to one that is optimal for the specified model and search space.

## replicates

Sometimes, you want to run the same condition multiple times, e.g., to make you less exposed to experimental noise and errors during subsequent data analysis. This can be done by setting the `replicates` parameter.
The replicates are added to the design, after it has been optimized.

Note that the design itself could include replicated conditions if that is most optimal for the specified model and search space.

In [9]:
number_of_experiments = 6

design, factor_names = get_optimal_DOE(factor_space,
                                       number_of_experiments,
                                       design_type='linear',
                                       replicates=2,
                                       )
print_friendly_design = np.asarray(design, dtype=str)

print("Factor names:")
print(factor_names)
print("Design:")
print(print_friendly_design)

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[['40.0' '100' '30' 'a']
 ['10.0' '20' '30' 'a']
 ['40.0' '20' '-30' 'b']
 ['10.0' '100' '30' 'b']
 ['10.0' '100' '-30' 'a']
 ['40.0' '100' '-30' 'b']
 ['40.0' '100' '30' 'a']
 ['10.0' '20' '30' 'a']
 ['40.0' '20' '-30' 'b']
 ['10.0' '100' '30' 'b']
 ['10.0' '100' '-30' 'a']
 ['40.0' '100' '-30' 'b']]


Note how we now have 6 treatments duplicated in the design for a total of 12 experiments.

## sorting

The treatments can be sorted in various ways. The default is to not sort the treatments. 

The following keywords can be used to specify sorting:
- `False`: No sorting performed
- `'ascending'`: Data points is sorted in ascending order. This is done after replicates are made effectively grouping replicates
- `'randomized'`: All data points are randomized in order.
- `'random_but_group_replicates'`: Datapoints are randomized, but replicates are kept together.

In [10]:
number_of_experiments = 6

design, factor_names = get_optimal_DOE(factor_space,
                                       number_of_experiments,
                                       design_type='linear',
                                       sorting='ascending',
                                       )
print_friendly_design = np.asarray(design, dtype=str)

print("Factor names:")
print(factor_names)
print("Design:")
print(print_friendly_design)

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[['10.0' '20' '-30' 'b']
 ['10.0' '20' '30' 'a']
 ['10.0' '100' '-30' 'a']
 ['40.0' '20' '-30' 'a']
 ['40.0' '20' '-30' 'b']
 ['40.0' '100' '30' 'b']]


In [11]:
number_of_experiments = 6

design, factor_names = get_optimal_DOE(factor_space,
                                       number_of_experiments,
                                       design_type='linear',
                                       sorting='random_but_group_replicates',
                                       replicates=3,
                                       )
print_friendly_design = np.asarray(design, dtype=str)

print("Factor names:")
print(factor_names)
print("Design:")
print(print_friendly_design)

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[['10.0' '100' '30' 'b']
 ['10.0' '100' '30' 'b']
 ['10.0' '100' '30' 'b']
 ['40.0' '20' '-30' 'b']
 ['40.0' '20' '-30' 'b']
 ['40.0' '20' '-30' 'b']
 ['10.0' '100' '-30' 'b']
 ['10.0' '100' '-30' 'b']
 ['10.0' '100' '-30' 'b']
 ['40.0' '100' '-30' 'a']
 ['40.0' '100' '-30' 'a']
 ['40.0' '100' '-30' 'a']
 ['40.0' '20' '30' 'a']
 ['40.0' '20' '30' 'a']
 ['40.0' '20' '30' 'a']
 ['10.0' '20' '-30' 'a']
 ['10.0' '20' '-30' 'a']
 ['10.0' '20' '-30' 'a']]


## model

Finally, we can also specify the model that we want to be able to fit to the data. This is done using the `model` parameter. Note that for this to work, the `design_type` parameter must not be set.

The model is specified as a string following the [`patsy` format](https://patsy.readthedocs.io/en/latest/formulas.html).

In this case, we could decide that we are not interested in interaction effects containing `var_x3` or `var_x4` but would like all main and quadratic effects along with the interaction between `var_x1` and `var_x2`.

In [12]:
model = 'var_x1 + var_x2 + var_x3 + var_x4 + var_x1:var_x2 + pow(var_x1, 2) + pow(var_x2, 2) + pow(var_x3, 2) + pow(var_x4, 2)'

number_of_experiments = 12

design, factor_names = get_optimal_DOE(factor_space,
                                        number_of_experiments,
                                        model=model,
                                        res=5
                                        )

print_friendly_design = np.asarray(design, dtype=str)

print("Factor names:")
print(factor_names)
print("Design:")
print(print_friendly_design)

Factor names:
['var_x1', 'var_x2', 'var_x3', 'var_x4']
Design:
[['40.0' '20' '-30' 'a']
 ['25.0' '20' '30' 'b']
 ['10.0' '20' '0' 'a']
 ['40.0' '100' '-30' 'b']
 ['40.0' '60' '0' 'b']
 ['10.0' '20' '-30' 'b']
 ['25.0' '60' '-30' 'a']
 ['25.0' '100' '0' 'a']
 ['10.0' '60' '30' 'a']
 ['10.0' '100' '-30' 'b']
 ['40.0' '100' '30' 'a']
 ['25.0' '100' '0' 'a']]


You can also get the factor names directly from the `Space` object.

In [13]:
factor_names = factor_space.names
my_model = f'{factor_names[0]} + {factor_names[1]} + {factor_names[2]} + {factor_names[0]}:{factor_names[1]} + {factor_names[0]}:{factor_names[2]} + {factor_names[1]}:{factor_names[2]}'
print(my_model)

var_x1 + var_x2 + var_x3 + var_x1:var_x2 + var_x1:var_x3 + var_x2:var_x3


## Log transformation

By using the `Space` class, a log transformation can be applied to `Real` dimensions.

In [29]:
factor_space_log = Space(dimensions=[
    Real(1, 100, name='var_x1', prior='log-uniform'),
    Integer(20, 100, name='var_x2'),
    Integer(-30, 30, name='var_x3'),
    Categorical(['a', 'b'], name='var_x4')
    ])

In [31]:
number_of_experiments = 24

design, factor_names = get_optimal_DOE(factor_space_log, number_of_experiments, design_type='response')
print_friendly_design = np.asarray(design, dtype=str)

print("Design:")
print(print_friendly_design)

Design:
[['100.0' '100' '-30' 'a']
 ['100.0' '60' '30' 'a']
 ['100.0' '20' '0' 'a']
 ['1.0' '100' '-30' 'a']
 ['10.0' '100' '0' 'a']
 ['10.0' '60' '-30' 'b']
 ['1.0' '20' '30' 'a']
 ['1.0' '20' '-30' 'a']
 ['100.0' '20' '-30' 'a']
 ['10.0' '60' '-30' 'a']
 ['1.0' '100' '30' 'b']
 ['1.0' '20' '30' 'b']
 ['1.0' '100' '-30' 'b']
 ['1.0' '20' '-30' 'b']
 ['1.0' '60' '0' 'a']
 ['100.0' '100' '30' 'b']
 ['100.0' '20' '-30' 'b']
 ['100.0' '100' '-30' 'b']
 ['100.0' '100' '30' 'a']
 ['100.0' '20' '30' 'b']
 ['1.0' '60' '0' 'b']
 ['15.848931924611142' '20' '30' 'a']
 ['1.0' '100' '30' 'a']
 ['10.0' '100' '0' 'b']]


Notice that the center value along the first dimension is now 10 rather than 50.5, as it would be on a linear scale.